In [5]:
note =  """
        secara overall ini sudah oke logicnya tinggal di ubah ke script python
        dan di rapihkan sama dibuat OOP dalam python 
        - dibuat automation 
        - runing di dockernya 
        - buat docker file dan docker composenya
        - mungki di tambahkan lognya juga ini 

        - kalo udah selesai prosesnya tinggal di submit di githubnya 
        - dan membuat read.me step by step setup sampai berhasil di komputer lain
        - boleh pake ai buak bikin codenya tapi harus paham konteksnya 


    

        """

In [6]:
import pandas as pd
import os

In [7]:
taxi_data_csv = "data/raw/taxi_zone_lookup.csv"
zone_df = pd.read_csv(taxi_data_csv)

# mengubah kolom taxi data_csv jadi snake_case
zone_df.columns = [ "location_id", "borough", "zone", "service_zone"]

zone_df.head()

print(zone_df['location_id'].is_unique)  # Harus True
print(zone_df['location_id'].isna().sum()) # Harus 0

True
0


In [8]:
# taxi data par
taxi_data_par= "data/raw/yellow_tripdata_2026-01.parquet"
df_par = pd.read_parquet(taxi_data_par)

# check rentang waktu data
print(df_par["tpep_pickup_datetime"].min())
print(df_par["tpep_pickup_datetime"].max())

df_par.info()

2025-12-31 23:57:29
2026-02-01 00:45:01
<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64      

In [9]:
# mengubah kolom taxi data_csv jadi snake_case
df_par =df_par.rename(columns = {
    "VendorID" : "vendor_id",
    "RatecodeID" : "rate_code_id",
    "tpep_pickup_datetime" : "pickup_datetime",
    "tpep_dropoff_datetime" : "dropoff_datetime",
    "PULocationID" : "pu_location_id",
    "DOLocationID" : "do_location_id",
    "Airport_fee" : "airport_fee"
    }
)   

# 
df_par.info()

<class 'pandas.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   vendor_id              int32         
 1   pickup_datetime        datetime64[us]
 2   dropoff_datetime       datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   rate_code_id           float64       
 6   store_and_fwd_flag     str           
 7   pu_location_id         int32         
 8   do_location_id         int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee            float64   

In [10]:
# mengubah time
df_par['pickup_hour'] = df_par['pickup_datetime'].dt.hour
df_par['pickup_date'] = df_par['pickup_datetime'].dt.date
df_par['pickup_day_name'] = df_par['pickup_datetime'].dt.day_name()

# trip_duration_minutes
df_par['trip_duration_minutes'] = ((df_par['dropoff_datetime'] - df_par['pickup_datetime']).dt.total_seconds() / 60).round(1)

# is_weekend
df_par['is_weekend'] = df_par['pickup_datetime'].dt.weekday >= 5  


In [11]:

#buat fungsi mappingnya time period
def time_period(hour):
    if 7 <= hour <= 9:
        return 'Morning Rush'
    elif 0 <= hour <= 5:
        return 'Late Night'
    elif 6 <= hour <=10:
        return 'Morning'
    elif 11 <= hour <=15:
        return 'Afternoon'
    elif 16 <= hour <= 19:
        return 'Evening Rush'
    elif 20 <= hour <= 23:
        return 'Night'
    else:
       return 'Invalid'
    
# kolom time_period
df_par['time_period'] = df_par['pickup_hour'].apply(time_period)



In [12]:
# payment type
def map_payment_type(x):
    if x == 1:
        return 'Credit Card'
    elif x == 2:
        return 'Cash'
    elif x == 3:
        return 'No Charge'
    elif x == 4:
        return 'Dispute'
    elif x == 0:
        return 'Unknown'
    else:
       return 'Invalid'
    
# kolom payment type
df_par['payment_type'] = df_par['payment_type'].apply(map_payment_type)


In [13]:
def map_store_fwd(x):
    if x == 'Y':
        return 'Store and Forward'
    elif x == 'N':
        return 'Normal'
    else:
        return 'Invalid'
    
# kolom payment type
df_par['store_and_fwd_flag'] = df_par['store_and_fwd_flag'].apply(map_store_fwd)

In [14]:
# PU location
df_par = df_par.merge(
    zone_df[['location_id', 'zone', 'borough', 'service_zone']],
    left_on='pu_location_id',
    right_on='location_id',
    how='left'
)

# Rename kolom hasil merge agar tidak membingungkan dengan dropoff nanti
df_par.rename(columns={
    ''
    'zone': 'pu_zone',
    'borough': 'pu_borough',
    'service_zone': 'pu_service_zone'
}, inplace=True)

df_par.drop(columns=['location_id'], inplace=True)


In [15]:
df_par = df_par.merge(
    zone_df[['location_id', 'zone', 'borough', 'service_zone']],
    left_on='do_location_id',
    right_on='location_id',
    how='left'
)

# Rename kolom hasil merge agar tidak membingungkan dengan dropoff nanti
df_par.rename(columns={
    ''
    'zone': 'do_zone',
    'borough': 'do_borough',
    'service_zone': 'do_service_zone'
}, inplace=True)

df_par.drop(columns=['location_id'], inplace=True)


In [16]:
final_column_order = [
    # ID & Waktu
    'vendor_id',
    'pickup_datetime',
    'dropoff_datetime',
    'trip_duration_minutes',
    
    # Waktu Ekstrak
    'pickup_date',
    'pickup_hour',
    'pickup_day_name',
    'is_weekend',
    'time_period',
    
    # Perjalanan
    'passenger_count',
    'trip_distance',
    'rate_code_id',
    'store_and_fwd_flag',
    
    # Lokasi Pickup (ID + detail zona)
    'pu_location_id',
    'pu_zone',
    'pu_borough',
    'pu_service_zone',   
    
    # Lokasi Dropoff (ID + detail zona)
    'do_location_id',
    'do_zone',
    'do_borough',
    'do_service_zone',   
    
    # Pembayaran
    'payment_type',
    
    # Rincian Biaya
    'fare_amount',
    'extra',
    'mta_tax',
    'tip_amount',
    'tolls_amount',
    'improvement_surcharge',
    'congestion_surcharge',
    'airport_fee',
    'cbd_congestion_fee',
    
    # Total
    'total_amount'
]

# Ambil kolom yang benar-benar tersedia di df_par
available_cols = [col for col in final_column_order if col in df_par.columns]

# Atur urutan
df_par = df_par[available_cols]

# Tampilkan beberapa baris pertama
df_par.head()


,vendor_id,pickup_datetime,dropoff_datetime,trip_duration_minutes,pickup_date,pickup_hour,pickup_day_name,is_weekend,time_period,passenger_count,...,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,airport_fee,cbd_congestion_fee,total_amount
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,5.6,2026-01-01,0,Thursday,False,Late Night,1.0,...,7.2,1.00,0.5,3.66,0.0,1.0,2.5,0.0,0.00,15.86
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,5.7,2026-01-01,0,Thursday,False,Late Night,0.0,...,7.9,4.25,0.5,0.00,0.0,1.0,2.5,0.0,0.75,13.65
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,8.9,2026-01-01,0,Thursday,False,Late Night,0.0,...,10.7,4.25,0.5,2.50,0.0,1.0,2.5,0.0,0.75,18.95
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,42.8,2026-01-01,0,Thursday,False,Late Night,4.0,...,38.7,1.00,0.5,11.11,0.0,1.0,2.5,0.0,0.75,55.56
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,13.5,2026-01-01,0,Thursday,False,Late Night,0.0,...,13.5,1.00,0.5,3.85,0.0,1.0,2.5,0.0,0.75,23.10


In [20]:
# buat kolom extract nya

import os

# Tentukan path folder penyimpanan
output_dir = './data/transformed'
os.makedirs(output_dir, exist_ok=True)  # buat folder jika belum ada

# Simpan sebagai Parquet (kompresi default snappy)
df_par.to_parquet(f'{output_dir}/trips_transformed.parquet_2', index=False)

print(f"File berhasil disimpan di {output_dir}/trips_transformed.parquet")

File berhasil disimpan di ./data/transformed/trips_transformed.parquet


In [21]:
# Path file parquet hasil transformasi
parquet_path = './data/transformed/taxi_trips_transformed2.parquet'
# Direktori tujuan
output_dir = './data/mart'
os.makedirs(output_dir, exist_ok=True)
csv_path = os.path.join(output_dir, 'data_mart_taxi_nyc.csv')

# Baca parquet
df = pd.read_parquet(parquet_path)

# Simpan sebagai CSV (tanpa index)
df.to_csv(csv_path, index=False)

print(f"File CSV berhasil disimpan di: {csv_path}")

FileNotFoundError: [Errno 2] No such file or directory: './data/transformed/taxi_trips_transformed2.parquet'

In [ ]:
# df_par[['passenger_count','fare_amount','trip_distance','extra','mta_tax','tip_amount','tolls_amount','improvement_surcharge','congestion_surcharge','cbd_congestion_fee','total_amount']].head(10)

# check value yang kosong nanti di tanyakan aja sama kak rio gimana enaknya
df_par.isna().sum()


,passenger_count,fare_amount,trip_distance,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,cbd_congestion_fee,total_amount
0,1.0,7.2,0.97,1.00,0.5,3.66,0.0,1.0,2.5,0.00,15.86
1,0.0,7.9,0.90,4.25,0.5,0.00,0.0,1.0,2.5,0.75,13.65
2,0.0,10.7,1.40,4.25,0.5,2.50,0.0,1.0,2.5,0.75,18.95
3,4.0,38.7,5.58,1.00,0.5,11.11,0.0,1.0,2.5,0.75,55.56
4,0.0,13.5,2.16,1.00,0.5,3.85,0.0,1.0,2.5,0.75,23.10
5,2.0,14.2,2.33,1.00,0.5,4.99,0.0,1.0,2.5,0.75,24.94
6,1.0,11.4,1.30,4.25,0.5,0.00,0.0,1.0,2.5,0.75,17.15
7,0.0,22.6,2.90,4.25,0.5,5.65,0.0,1.0,2.5,0.75,34.00
8,1.0,37.3,5.34,1.00,0.5,8.61,0.0,1.0,2.5,0.75,51.66
9,3.0,10.7,1.83,1.00,0.5,2.36,0.0,1.0,2.5,0.00,18.06


In [ ]:

# Asumsikan df_par sudah dimuat dengan kolom:
# - pickup_datetime (datetime)
# - dropoff_datetime (datetime)
# - trip_distance (float)
# - mungkin sudah ada trip_duration_minutes

# Pastikan kolom durasi (dalam menit) tersedia, atau hitung ulang
if 'trip_duration_minutes' not in df_par.columns:
    df_par['trip_duration_minutes'] = (df_par['dropoff_datetime'] - df_par['pickup_datetime']).dt.total_seconds() / 60

# Inisialisasi kolom error_type dengan string kosong
df_par['error_type'] = ''

# Kondisi 1: durasi tidak valid (pickup >= dropoff atau durasi <= 0)
invalid_duration_mask = (df_par['pickup_datetime'] >= df_par['dropoff_datetime']) | (df_par['trip_duration_minutes'] <= 0)

# Kondisi 2: jarak tidak valid (trip_distance <= 0)
invalid_distance_mask = (df_par['trip_distance'] <= 0)

# Isi error_type berdasarkan kondisi (prioritas bisa digabung)
df_par.loc[invalid_duration_mask, 'error_type'] = 'duration invalid'
df_par.loc[invalid_distance_mask, 'error_type'] = df_par.loc[invalid_distance_mask, 'error_type'] + '; distance invalid' 
# Perbaiki agar tidak double separator
df_par.loc[invalid_distance_mask & ~invalid_duration_mask, 'error_type'] = 'distance invalid'
df_par.loc[invalid_duration_mask & invalid_distance_mask, 'error_type'] = 'duration invalid; distance invalid'

# Tentukan data valid (tidak memiliki error)
df_valid = df_par[df_par['error_type'] == ''].copy()
df_invalid = df_par[df_par['error_type'] != ''].copy()

# Hapus kolom error_type dari data valid (opsional)
df_valid.drop(columns=['error_type'], inplace=True)

# Buat folder output
output_dir = './data/mart_cleaned'
os.makedirs(output_dir, exist_ok=True)

# Simpan data valid ke folder mart_cleaned (format parquet untuk efisiensi)
df_valid.to_parquet(f'{output_dir}/cleaned_trips.parquet', index=False)

# Simpan data invalid ke file karantina (CSV agar mudah dibaca)
df_invalid.to_csv(f'{output_dir}/quarantine_trips.csv', index=False)

# Laporan sederhana
total_rows = len(df_par)
valid_rows = len(df_valid)
invalid_rows = len(df_invalid)
duration_invalid = len(df_par[invalid_duration_mask])
distance_invalid = len(df_par[invalid_distance_mask])
both_invalid = len(df_par[invalid_duration_mask & invalid_distance_mask])

print("=== LAPORAN KUALITAS DATA ===")
print(f"Total baris: {total_rows:,}")
print(f"Data valid: {valid_rows:,} ({valid_rows/total_rows:.2%})")
print(f"Data invalid (karantina): {invalid_rows:,} ({invalid_rows/total_rows:.2%})")
print("\nRincian error:")
print(f"- Durasi tidak valid (pickup >= dropoff atau durasi <= 0): {duration_invalid:,}")
print(f"- Jarak tidak valid (trip_distance <= 0): {distance_invalid:,}")
print(f"- Keduanya: {both_invalid:,}")
print(f"\nFile disimpan di: {output_dir}")
print("- cleaned_trips.parquet (data valid)")
print("- quarantine_trips.csv (data invalid dengan kolom error_type)")

In [ ]:
#Output nya bakal kayak gini sih 

./data/mart_cleaned/
├── cleaned_trips.parquet     # Data valid (tanpa error_type)
└── quarantine_trips.csv      # Data invalid dengan kolom 'error_type'

SyntaxError: invalid character '├' (U+251C) (3632210138.py, line 3)